# Planet SDK 101: Lake Lagunita Search, Ordering, and NDVI

![Historical photograph of Lake Lagunita on the Stanford campus, with a prompt-style note: show a broad, shallow campus lake surrounded by open grass and trees, useful as context for a satellite imagery lab.](https://news.stanford.edu/wp-content/uploads/2020/02/CC154.jpg "Lake Lagunita, circa 1903-1906")

This notebook repeats the workflow from `planet_API_101_LakeLagunita.ipynb`, but it uses the official Planet Python SDK wherever possible instead of asking you to build REST API requests by hand.

## What You Should Understand

By the end of this notebook, you should be able to:

- Authenticate to Planet from Google Colab using Secret Manager.
- Load and map a GeoJSON area of interest, or AOI.
- Build Planet SDK filters for location, date, and metadata such as cloud or clear percentage.
- Search for PlanetScope scenes and inspect the returned metadata.
- Visualize scene footprints over the AOI.
- Sort scenes by quality and reduce them to the smallest useful set for AOI coverage.
- Create an Orders API request with clip and NDVI band math tools.
- Submit, monitor, download, and visualize ordered imagery when you are ready.

## Dataset Notes

This lab uses `data/lakelagunita.geojson` as the AOI. The file is a small polygon around Lake Lagunita on the Stanford campus. The satellite imagery is searched from Planet's PlanetScope `PSScene` catalog through the Planet Data API and ordered through the Planet Orders API.


## Installing

Colab does not always include the Planet SDK or geospatial raster libraries by default. This cell installs the packages used in the notebook.

- `planet` is Planet's official Python SDK for searching data and creating orders.
- `rasterio` reads downloaded GeoTIFF imagery.
- `folium` makes interactive web maps inside the notebook.
- `shapely` handles polygon overlap calculations.
- `matplotlib` displays downloaded raster images.
- `nest_asyncio` helps asynchronous SDK calls run smoothly inside notebooks.


In [ ]:
# The exclamation mark tells a notebook to run a shell command instead of Python code.
# We use -q for "quiet" so the installation output is shorter and easier to read.
!pip -q install planet rasterio folium shapely matplotlib nest_asyncio


## Importing

Python packages are collections of code written by other people. Importing a package makes its functions available in this notebook.

Planet's SDK uses asynchronous Python for network requests. In practice, that means we will use `async`, `await`, and `async with` when we search, order, monitor, and download data.


In [ ]:
# Standard library imports come with Python.
# They help us work with files, dates, JSON data, and asynchronous tasks.
import asyncio
import glob
import json
import os
from datetime import datetime, timedelta
from pathlib import Path

# matplotlib is used later to draw downloaded rasters in the notebook.
import matplotlib.pyplot as plt

# nest_asyncio lets Colab reuse its existing event loop for async SDK calls.
import nest_asyncio

# rasterio reads geospatial raster files, such as GeoTIFFs downloaded from Planet.
import rasterio

# folium creates interactive Leaflet maps that display in notebook cells.
import folium

# shapely turns GeoJSON-like dictionaries into geometry objects we can compare.
from shapely.geometry import shape
from shapely.ops import unary_union

# Planet SDK imports.
# Auth stores the API key, Session manages authenticated connections,
# OrdersClient talks to the Orders API, order_request builds order JSON,
# and data_filter builds Data API search filters.
from planet import Auth, OrdersClient, Session, data_filter, order_request, reporting

# Colab runs an event loop behind the scenes. This makes repeated await calls work better.
nest_asyncio.apply()

# This helper prints nested dictionaries in a readable way.
def indent(data):
    """Print JSON-like Python data with indentation for easier reading."""
    print(json.dumps(data, indent=2))


## Authenticating

In Colab, store your Planet API key in Secret Manager with the name `PL_API_KEY`, then toggle notebook access on for that secret.

This cell reads the secret without printing it. It also stores the key in the environment variable names that Planet examples commonly use.


In [ ]:
# userdata is Colab's interface to Secret Manager.
# This import only works inside Google Colab.
from google.colab import userdata

# Read the Planet API key from Colab Secret Manager.
# The secret should be named PL_API_KEY.
API_KEY = userdata.get("PL_API_KEY")

# Stop early with a helpful message if the secret is missing.
if not API_KEY:
    raise ValueError(
        "No PL_API_KEY secret was found. Add your Planet API key to Colab Secret Manager "
        "and enable notebook access for this notebook."
    )

# The Planet SDK can read credentials from environment variables.
# PL_API_KEY is the main name used by the current Planet Python client.
os.environ["PL_API_KEY"] = API_KEY

# PLANET_API_KEY is also set because older examples sometimes expect this name.
os.environ["PLANET_API_KEY"] = API_KEY

# Build an Auth object so we can confirm that the key was loaded.
# We do not print the key itself, because API keys should stay private.
planet_auth = Auth.from_key(API_KEY)

print("Planet API key loaded from Colab Secret Manager.")


## Testing

Before doing a full search, run a tiny SDK search. This confirms that authentication works and that the SDK can reach Planet's services.


In [ ]:
async def test_planet_connection():
    """Run the smallest useful SDK search to confirm that authentication works."""
    # Limit the search to one item so the test is quick and low-noise.
    item_types = ["PSScene"]

    # A broad date filter is enough for a connection test.
    test_filter = data_filter.date_range_filter(
        "acquired",
        gt=datetime(year=2023, month=1, day=1),
        lt=datetime(year=2023, month=1, day=2),
    )

    # Session opens an authenticated SDK connection.
    async with Session() as sess:
        # The data client searches Planet's imagery catalog.
        data_client = sess.client("data")

        # SDK searches return an async iterator, so we use async for.
        results = [item async for item in data_client.search(
            search_filter=test_filter,
            item_types=item_types,
            limit=1,
        )]

    # We only need to know that the request completed.
    print(f"Connection test complete. Test search returned {len(results)} item(s).")

await test_planet_connection()


## AOI With Visualization

An AOI, or area of interest, tells the search where to look. Here the AOI is a small GeoJSON polygon around Lake Lagunita.

![Placeholder map illustration: show a simple satellite map of Stanford with a highlighted rectangular polygon around Lake Lagunita. The image should make clear that an AOI is a search boundary, not the final image footprint.](attachment:lake-lagunita-aoi-placeholder.png)


In [ ]:
# The notebook may be run from the repository root or from the data folder.
# These candidate paths let the same notebook work in either situation.
aoi_candidates = [
    Path("data/lakelagunita.geojson"),
    Path("lakelagunita.geojson"),
]

# Find the first AOI path that exists.
aoi_path = next((path for path in aoi_candidates if path.exists()), None)

# Stop with a clear message if the AOI file is not available.
if aoi_path is None:
    raise FileNotFoundError(
        "Could not find lakelagunita.geojson. Put data/lakelagunita.geojson in the runtime, "
        "or run this notebook from the repository root."
    )

# Load the GeoJSON text into a Python dictionary.
with aoi_path.open() as f:
    aoi_geojson = json.load(f)

# The Planet Data API filter needs a single geometry dictionary.
# This notebook uses the first feature in the FeatureCollection.
aoi_geometry = aoi_geojson["features"][0]["geometry"]

# Shapely geometry is useful for overlap and coverage calculations later.
aoi_shape = shape(aoi_geometry)

# Calculate the center of the AOI so the map opens in the right place.
aoi_center_lat = aoi_shape.centroid.y
aoi_center_lon = aoi_shape.centroid.x

print(f"Loaded AOI from {aoi_path}")
print(f"AOI center: {aoi_center_lat:.6f}, {aoi_center_lon:.6f}")


In [ ]:
# Create a Folium map centered on the AOI.
aoi_map = folium.Map(
    location=[aoi_center_lat, aoi_center_lon],
    zoom_start=15,
    tiles="cartodbpositron",
)

# Add the AOI polygon to the map.
folium.GeoJson(
    aoi_geojson,
    name="Lake Lagunita AOI",
    style_function=lambda feature: {
        "color": "#0072B2",
        "weight": 3,
        "fillColor": "#56B4E9",
        "fillOpacity": 0.25,
    },
).add_to(aoi_map)

# LayerControl lets students toggle map layers on and off.
folium.LayerControl().add_to(aoi_map)

aoi_map


## Search

The source notebook searches for PlanetScope scenes over Lake Lagunita during the 2022-2023 precipitation season. We will keep those same basic search parameters:

- Item type: `PSScene`
- Start date: December 10, 2022
- End date: September 30, 2023
- AOI: `data/lakelagunita.geojson`

The SDK's `data_filter` helpers build the filter dictionaries for us.


In [ ]:
# Planet item types identify different catalogs of imagery.
# PSScene means PlanetScope scene imagery.
item_types = ["PSScene"]

# Date settings copied from the source workflow.
search_start = datetime(year=2022, month=12, day=10)
search_end = datetime(year=2023, month=9, day=30)

# The geometry filter keeps only scenes that intersect the Lake Lagunita AOI.
geometry_filter = data_filter.geometry_filter(aoi_geometry)

# The date filter keeps only scenes acquired during the target time window.
date_filter = data_filter.date_range_filter(
    "acquired",
    gt=search_start,
    lt=search_end,
)

# For the first search, combine only AOI and date.
# We add metadata filtering for clouds in a later section.
base_search_filter = data_filter.and_filter([
    geometry_filter,
    date_filter,
])

# Printing the filter helps reveal that SDK helper functions are building normal dictionaries.
indent(base_search_filter)


In [ ]:
async def search_planet_items(search_filter, limit=500):
    """Search Planet with the SDK and return a normal Python list of item dictionaries."""
    async with Session() as sess:
        # Create the Data API client from the authenticated session.
        data_client = sess.client("data")

        # The SDK returns an async stream of items.
        # Collecting the stream into a list makes later teaching cells easier to inspect.
        items = [item async for item in data_client.search(
            search_filter=search_filter,
            item_types=item_types,
            limit=limit,
        )]

    return items

# Run the base search.
base_items = await search_planet_items(base_search_filter, limit=500)

print(f"Base search returned {len(base_items)} PlanetScope scene(s).")


## Report Results

A search result is a list of item dictionaries. Each item has an `id`, a `geometry`, and a `properties` dictionary with metadata such as acquisition time and clear percentage.


In [ ]:
# Print a compact report for the first several items.
# The [:10] slice means "show only the first 10 items."
for item in base_items[:10]:
    properties = item["properties"]
    print(
        item["id"],
        "acquired:", properties.get("acquired"),
        "clear_percent:", properties.get("clear_percent"),
        "cloud_cover:", properties.get("cloud_cover"),
    )

# If there are more than 10 results, tell students the list was shortened.
if len(base_items) > 10:
    print(f"... {len(base_items) - 10} more item(s) not printed here.")


## Metadata

Metadata is descriptive information about an image. For remote sensing searches, metadata helps you decide whether a scene is useful before ordering the actual raster data.

Common metadata fields include acquisition date, cloud cover, clear percentage, sun angle, satellite ID, and ground sample distance.


In [ ]:
# Choose one item to inspect closely.
# Checking first avoids a confusing index error if the search returned no scenes.
if not base_items:
    raise ValueError("The base search returned no items. Check the AOI, dates, or Planet access permissions.")

example_item = base_items[0]
example_properties = example_item["properties"]

print("Example item ID:", example_item["id"])
print("Available metadata keys:")

# sorted() prints keys in alphabetical order, which makes them easier to scan.
for key in sorted(example_properties.keys()):
    print("-", key)


In [ ]:
# Print the full example item for students who want to see the nested structure.
# This is often the easiest way to discover useful metadata fields for future filters.
indent(example_item)


## Footprints Visualization

A scene footprint is the polygon showing the ground area covered by one satellite scene. Footprints are useful because the AOI may be smaller than a full image, and multiple scenes may overlap the AOI.


In [ ]:
# Create a folder for small derived files from this notebook.
output_dir = Path("output_planet_sdk_lakelagunita")
output_dir.mkdir(exist_ok=True)

# Convert search results into a GeoJSON FeatureCollection of scene footprints.
scene_footprints = {
    "type": "FeatureCollection",
    "features": [],
}

for item in base_items:
    # Each feature keeps the Planet item properties and the scene geometry.
    scene_footprints["features"].append({
        "type": "Feature",
        "properties": item["properties"],
        "geometry": item["geometry"],
    })

footprints_path = output_dir / "base_search_footprints.geojson"

# Save the footprints so they can be reused or inspected outside the notebook.
with footprints_path.open("w") as f:
    json.dump(scene_footprints, f)

print(f"Saved {len(scene_footprints['features'])} footprint(s) to {footprints_path}")


In [ ]:
# Make a map that shows the AOI and the scene footprints together.
footprint_map = folium.Map(
    location=[aoi_center_lat, aoi_center_lon],
    zoom_start=12,
    tiles="cartodbpositron",
)

# Draw the AOI in blue.
folium.GeoJson(
    aoi_geojson,
    name="Lake Lagunita AOI",
    style_function=lambda feature: {
        "color": "#0072B2",
        "weight": 3,
        "fillColor": "#56B4E9",
        "fillOpacity": 0.25,
    },
).add_to(footprint_map)

# Draw scene footprints in orange.
folium.GeoJson(
    scene_footprints,
    name="PlanetScope scene footprints",
    style_function=lambda feature: {
        "color": "#D55E00",
        "weight": 1,
        "fillOpacity": 0.05,
    },
).add_to(footprint_map)

folium.LayerControl().add_to(footprint_map)

footprint_map


## Filtering: AOI, Date, and Metadata Clouds

The first search used only AOI and date filters. Now we add a metadata filter for `clear_percent`, following the source notebook's threshold of greater than 80.

Planet metadata can describe clouds in more than one way. This notebook uses `clear_percent` because the source workflow used it and because it is easy to interpret: higher values usually mean less cloud obstruction.


In [ ]:
# Keep scenes where Planet estimates that more than 80 percent is clear.
clear_percent_filter = data_filter.range_filter("clear_percent", gt=80)

# Combine location, date, and metadata filters into one SDK filter.
clear_search_filter = data_filter.and_filter([
    geometry_filter,
    date_filter,
    clear_percent_filter,
])

# Search again with the stronger filter.
clear_items = await search_planet_items(clear_search_filter, limit=500)

print(f"Base AOI/date search returned {len(base_items)} item(s).")
print(f"AOI/date/clear_percent search returned {len(clear_items)} item(s).")


## Sorting Quality

Search filters remove scenes that do not meet minimum rules. Sorting ranks the scenes that remain. Here we sort by `clear_percent`, with the clearest scenes first.


In [ ]:
# Sort all filtered scenes from highest clear_percent to lowest clear_percent.
# The get(..., 0) pattern protects us if a rare item is missing that metadata field.
quality_sorted_items = sorted(
    clear_items,
    key=lambda item: item["properties"].get("clear_percent", 0),
    reverse=True,
)

# Print the clearest few scenes.
for item in quality_sorted_items[:10]:
    print(
        item["id"],
        item["properties"].get("acquired"),
        "clear_percent:", item["properties"].get("clear_percent"),
    )


## Reducing to Coverage

Ordering every matching scene can create unnecessary cost, download time, and clutter. The source workflow groups scenes into roughly 30-day windows, sorts each group by clarity, and keeps only scenes that add new AOI coverage.

This is a simple greedy method:

1. Sort scenes in each time window from clearest to least clear.
2. Add a scene if it covers part of the AOI that previous selected scenes did not cover.
3. Stop once the selected scenes cover nearly all of the AOI.


In [ ]:
def parse_planet_time(item):
    """Convert a Planet acquired timestamp string into a Python datetime."""
    # Planet timestamps look like 2023-02-07T18:05:04.123Z.
    return datetime.strptime(item["properties"]["acquired"], "%Y-%m-%dT%H:%M:%S.%fZ")


def group_items_by_days(items, days=30):
    """Group items into time windows measured from the first item in each group."""
    # Sort oldest to newest so groups follow calendar order.
    ordered_items = sorted(items, key=parse_planet_time)

    groups = []
    current_group = []
    group_start = None

    for item in ordered_items:
        item_time = parse_planet_time(item)

        # Start the first group.
        if group_start is None:
            group_start = item_time
            current_group = [item]
            continue

        # Keep adding to the group until the window is too wide.
        if item_time < group_start + timedelta(days=days):
            current_group.append(item)
        else:
            groups.append(current_group)
            current_group = [item]
            group_start = item_time

    # Add the final group after the loop ends.
    if current_group:
        groups.append(current_group)

    return groups

# Build approximately monthly groups from the filtered search results.
grouped_items = group_items_by_days(clear_items, days=30)

print(f"Created {len(grouped_items)} time group(s).")
for index, group in enumerate(grouped_items, start=1):
    start_date = parse_planet_time(group[0]).date()
    end_date = parse_planet_time(group[-1]).date()
    print(f"Group {index}: {len(group)} item(s), {start_date} to {end_date}")


In [ ]:
def scene_aoi_overlap(item, aoi):
    """Return the part of a scene footprint that overlaps the AOI."""
    # item["geometry"] is GeoJSON-like geometry from Planet.
    scene_shape = shape(item["geometry"])

    # intersection keeps only the shared area between the scene and AOI.
    return scene_shape.intersection(aoi)


def reduce_group_to_coverage(group, aoi, target_fraction=0.99):
    """Select clear scenes that add new AOI coverage until the target is reached."""
    # Clearest scenes are considered first.
    sorted_group = sorted(
        group,
        key=lambda item: item["properties"].get("clear_percent", 0),
        reverse=True,
    )

    selected = []
    covered_geometry = None
    aoi_area = aoi.area

    for item in sorted_group:
        overlap = scene_aoi_overlap(item, aoi)

        # Skip scenes that do not touch the AOI.
        if overlap.is_empty or overlap.area == 0:
            continue

        # Union combines the new overlap with the coverage we already selected.
        candidate_coverage = overlap if covered_geometry is None else unary_union([covered_geometry, overlap])

        previous_area = 0 if covered_geometry is None else covered_geometry.area
        new_area = candidate_coverage.area

        # Keep the scene only if it adds measurable new coverage.
        if round(new_area, 10) > round(previous_area, 10):
            selected.append(item)
            covered_geometry = candidate_coverage

        # Stop once the selected scenes cover nearly all of the AOI.
        if covered_geometry is not None and (covered_geometry.area / aoi_area) >= target_fraction:
            break

    coverage_fraction = 0 if covered_geometry is None else covered_geometry.area / aoi_area

    return selected, coverage_fraction

# Reduce every time group to its smallest useful coverage set.
reduced_groups = []

for index, group in enumerate(grouped_items, start=1):
    selected, coverage_fraction = reduce_group_to_coverage(group, aoi_shape, target_fraction=0.99)

    if selected:
        reduced_groups.append(selected)

    print(
        f"Group {index}: reduced {len(group)} item(s) to {len(selected)} item(s); "
        f"AOI coverage is about {coverage_fraction:.1%}."
    )


In [ ]:
# The Orders API composites later items on top of earlier items.
# Sort each group from lower to higher clear_percent so the clearest item is placed last/on top.
order_item_groups = []

for group in reduced_groups:
    order_group = sorted(
        group,
        key=lambda item: item["properties"].get("clear_percent", 0),
    )
    order_item_groups.append(order_group)

# Report the final scenes that would be used in each order.
for index, group in enumerate(order_item_groups, start=1):
    print(f"Order group {index}")
    for item in group:
        print(
            " ",
            item["id"],
            item["properties"].get("acquired")[:10],
            "clear_percent:", item["properties"].get("clear_percent"),
        )


## Creating an Order

Searching finds candidate scenes. Ordering asks Planet to prepare downloadable files from those scenes.

The request below uses `order_request`, the SDK's helper module for building Orders API JSON. We request `analytic_udm2` assets for `PSScene` items, then apply tools before delivery.


## Tools: Clip and Band Math NDVI

Planet Orders tools process imagery on Planet's side before you download it.

- `clip` cuts each image down to the Lake Lagunita AOI.
- `band_math` creates a new raster band from a formula.
- `composite` merges scenes in an order group into one output image.

For a four-band PlanetScope analytic asset, a common band order is blue, green, red, near infrared. NDVI uses near infrared and red:

`NDVI = (NIR - Red) / (NIR + Red)`

The expression below scales NDVI into an 8-bit image with approximate values from 0 to 200, where values near 100 are around zero NDVI.


In [ ]:
def build_lakelagunita_order(order_name, item_ids):
    """Build a Planet Orders request for clipped NDVI composites."""
    # A product tells Planet which item IDs, asset bundle, and item type to order.
    products = [
        order_request.product(item_ids, "analytic_udm2", "PSScene")
    ]

    # Clip to the AOI so downloaded files are smaller and focused on the lab area.
    clip_tool = order_request.clip_tool(aoi=aoi_geometry)

    # NDVI for four-band PlanetScope analytic imagery commonly uses:
    # b4 = near infrared, b3 = red.
    # Multiplying by 100 and adding 100 stores the result in unsigned 8-bit form.
    ndvi_tool = order_request.band_math_tool(
        b1="(b4-b3)/(b4+b3)*100+100",
        pixel_type="8U",
    )

    # Composite combines multiple scenes into one output raster.
    composite_tool = order_request.composite_tool()

    # Tool order matters: clip first, calculate NDVI, then composite.
    tools = [clip_tool, ndvi_tool, composite_tool]

    # build_request creates the final order dictionary expected by the Orders API.
    return order_request.build_request(
        name=order_name,
        products=products,
        tools=tools,
    )


In [ ]:
# Build one order request per reduced time group.
order_requests = []
order_download_folders = []
order_name_prefix = "lake_lagunita_sdk_ndvi_"

for group in order_item_groups:
    # Use the first scene date in the group to make a readable order name.
    order_date = group[0]["properties"]["acquired"][:10]
    order_name = f"{order_name_prefix}{order_date}"

    # The Orders API needs item IDs, not the full item dictionaries.
    item_ids = [item["id"] for item in group]

    # Build and save the order request.
    order_requests.append(build_lakelagunita_order(order_name, item_ids))
    order_download_folders.append(order_name)

print(f"Prepared {len(order_requests)} order request(s).")

# Preview the first order request so students can inspect the JSON-like structure.
if order_requests:
    indent(order_requests[0])


## Ordering

Submitting an order may use Planet quota, require permissions, and take time. For teaching safety, the notebook does not submit orders until you set `SUBMIT_ORDERS = True`.

Run the preview cells first. When you are confident that the item IDs, tools, and AOI are correct, change the flag and run the ordering cells.


In [ ]:
# Safety switch:
# Leave this as False while learning, testing, or demonstrating the notebook.
# Change to True only when you are ready to submit real Planet orders.
SUBMIT_ORDERS = False

print("SUBMIT_ORDERS is set to", SUBMIT_ORDERS)


## Waiting for Order: Monitoring

The SDK can create an order and then wait for Planet to finish preparing it. Monitoring is important because downloads are not available immediately after submission.


In [ ]:
async def submit_monitor_download_order(request):
    """Create one order, wait for completion, and download the finished files."""
    async with Session() as sess:
        # OrdersClient manages create, wait, inspect, and download operations.
        orders_client = OrdersClient(sess)

        # Create the order on Planet's servers.
        order = await orders_client.create_order(request)
        order_id = order["id"]
        print(f"Created order {request['name']} with id {order_id}")

        # reporting.StateBar displays changing order states in notebook output.
        with reporting.StateBar(state="created", order_id=order_id) as bar:
            # max_attempts=0 means keep waiting until Planet returns a final state.
            await orders_client.wait(
                order_id,
                max_attempts=0,
                callback=bar.update_state,
            )

        # Download into a folder named after the order.
        download_dir = Path(request["name"])
        download_dir.mkdir(exist_ok=True)

        await orders_client.download_order(
            order_id,
            directory=str(download_dir),
        )

        print(f"Downloaded order {order_id} to {download_dir}")
        return order_id


In [ ]:
# This cell submits every prepared order only if the safety switch is True.
submitted_order_ids = []

if SUBMIT_ORDERS:
    # Gather lets multiple orders run concurrently instead of one at a time.
    submitted_order_ids = await asyncio.gather(*[
        submit_monitor_download_order(request)
        for request in order_requests
    ])

    print("Submitted order IDs:")
    for order_id in submitted_order_ids:
        print("-", order_id)
else:
    print("No orders were submitted because SUBMIT_ORDERS is False.")
    print("Review the order request above, then set SUBMIT_ORDERS = True when you are ready.")


## Downloading

The previous cell downloads completed orders automatically when `SUBMIT_ORDERS = True`. If you already submitted orders in another session, you can download them by ID with the helper below.


In [ ]:
async def download_existing_order(order_id, download_dir):
    """Download an existing Planet order by order ID."""
    async with Session() as sess:
        orders_client = OrdersClient(sess)

        # Create the local folder before downloading files into it.
        Path(download_dir).mkdir(exist_ok=True)

        # download_order retrieves all available files for the completed order.
        await orders_client.download_order(order_id, directory=str(download_dir))

    print(f"Downloaded existing order {order_id} to {download_dir}")

# Example use after you have a real completed order ID:
# await download_existing_order("PASTE_ORDER_ID_HERE", "lake_lagunita_existing_order")


## Visualization

After orders are downloaded, each order folder should contain a clipped, band-math, composited GeoTIFF. The source workflow looked for `composite.tif`; this cell follows the same pattern and maps the NDVI-style output with a green color ramp.

![Placeholder raster visualization: show several small NDVI maps of Lake Lagunita through time, with greener tones for higher vegetation signal and neutral tones for bare or water-covered areas.](attachment:lake-lagunita-ndvi-placeholder.png)


In [ ]:
# Search recursively for composite GeoTIFFs downloaded by Planet Orders.
# The ** pattern means "look through subfolders too."
composite_files = sorted(glob.glob("**/composite.tif", recursive=True))

print(f"Found {len(composite_files)} composite.tif file(s).")
for path in composite_files:
    print("-", path)


In [ ]:
if not composite_files:
    print("No downloaded composites found yet. Submit/download orders before running this visualization.")
else:
    # Keep the grid size flexible so it works with different numbers of orders.
    ncols = min(4, len(composite_files))
    nrows = (len(composite_files) + ncols - 1) // ncols

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(4 * ncols, 4 * nrows),
        squeeze=False,
    )

    for file_path, ax in zip(composite_files, axes.flatten()):
        # rasterio opens the GeoTIFF and reads the first band.
        with rasterio.open(file_path) as src:
            ndvi_scaled = src.read(1)

        # Values around 100 represent NDVI near zero after our scaling formula.
        image = ax.imshow(ndvi_scaled, cmap="RdYlGn", vmin=0, vmax=200)

        # Use the parent folder name as a simple title.
        ax.set_title(Path(file_path).parts[0])
        ax.axis("off")

    # Hide any unused axes in the final row.
    for ax in axes.flatten()[len(composite_files):]:
        ax.axis("off")

    fig.colorbar(image, ax=axes, shrink=0.75, label="Scaled NDVI: NDVI * 100 + 100")
    plt.show()


## Packaging Results for Colab Download

Colab runtimes are temporary. If you created outputs you want to keep, zip the order folders and download the zip file before ending the session.


In [ ]:
# This command zips the output folder and any downloaded Lake Lagunita order folders.
# It is safe to run after downloads have finished.
!zip -r /content/lake_lagunita_planet_sdk_outputs.zip output_planet_sdk_lakelagunita lake_lagunita_sdk_ndvi_* 2>/dev/null


In [ ]:
# Colab's files helper opens a browser download prompt for the zip file.
from google.colab import files

files.download("/content/lake_lagunita_planet_sdk_outputs.zip")
